# NER and POS baselines on Colab

Full fine-tuning and LoRA (r=8) on `Davlan/afro-xlmr-large-76L`, 6 languages (amh, hau, kin, lug, swa, zul), 3 seeds,
for two tasks: NER (MasakhaNER 2.0, Amharic from 1.0) and POS (MasakhaPOS, Amharic from UD_Amharic-ATT).
Each run trains on one source language and tests on all 6, so the 6x6 zero-shot transfer matrices come from the same runs.

**Before you start:** Runtime > Change runtime type > **L4 GPU** (the SIB-200 baseline also ran on L4; keep the same GPU).
Then Runtime > Run all.

**Time on an L4 (estimate):** NER about 4 hours (36 runs on 5k to 8k sentences each), POS about 1 hour.
NER runs first, then POS. The time-left estimate printed after every run will be more accurate after a few runs.

**If Colab disconnects:** reconnect and Run all again. Results are written to Google Drive after every run,
and finished runs are skipped.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4 GPU, then run again."

## 2. Mount Google Drive (results are saved here)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/typology-aware-peft")
TASKS = ["ner", "pos"]
OUT = {t: DRIVE / "results" / t for t in TASKS}
for p in OUT.values():
    p.mkdir(parents=True, exist_ok=True)
print("Results folders:", *OUT.values(), sep="\n  ")

## 3. Get the code (latest version from GitHub)

In [ ]:
import os, shutil
REPO_URL = "https://github.com/WanjaWhoopie/typology-aware-peft.git"
WORK = Path("/content/typology-aware-peft")
if WORK.exists():
    !git -C {WORK} pull -q
else:
    !git clone -q {REPO_URL} {WORK}
for name in ["token_baseline.py", "token_report.py", "sib200_report.py"]:
    assert (WORK / "src" / name).exists(), f"src/{name} is not on GitHub yet; push it first"
os.chdir(WORK)
!git log --oneline -1

## 4. Install packages

Same pinned versions as the SIB-200 run.

In [ ]:
!pip install -q transformers==4.57.2 datasets==4.4.1 peft==0.21.2 scikit-learn
import transformers, datasets, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| peft", peft.__version__)

## 5. Smoke test (a few minutes)

LoRA on the base model, 2 languages, 64 sentences, 1 epoch, for each task. Only checks that everything runs
(and downloads the data). Scores near zero and "collapsed" warnings are normal here.

In [ ]:
for t in TASKS:
    !python src/token_baseline.py --task {t} --smoke --model Davlan/afro-xlmr-base

## 6. Full NER run

* **Out of memory:** rerun this cell with `--bs 8` added (and write down that the batch size changed).
* **Collapsed run** (dev F1 below 0.2): finish the job, then use step 9.

In [ ]:
out = OUT["ner"]
!python src/token_baseline.py --task ner --out {out}

## 7. Full POS run

In [ ]:
out = OUT["pos"]
!python src/token_baseline.py --task pos --out {out}

## 8. Check the runs

In [ ]:
import json
def show(task):
    runs = [json.loads(l) for l in (OUT[task] / "runs.jsonl").read_text().splitlines() if l.strip()]
    print(f"{task.upper()}: {len(runs)} of 36 runs finished")
    print(f"{'method':6} {'seed':>4} {'src':4} {'dev':>6} {'epoch':>5} {'min':>5} {'GB':>5}  collapsed")
    for r in sorted(runs, key=lambda r: (r["method"], r["src"], r["seed"])):
        print(f"{r['method']:6} {r['seed']:>4} {r['src']:4} {r['best_dev']:>6.3f} {r['best_epoch']:>5} "
              f"{r['seconds'] / 60:>5.1f} {r['peak_mem_gb'] or 0:>5.1f}  {'YES' if r['collapsed'] else ''}")
    print()
    return runs
runs = {t: show(t) for t in TASKS}

## 9. Optional: redo collapsed runs with a lower learning rate

Only if step 8 shows collapsed runs. Set `REDO = True`, run this cell, then run step 8 again.
It removes the collapsed runs from the results files and trains them again with half the learning rate.
The new learning rate is stored in `runs.jsonl`, so mention it in the write-up.

In [ ]:
REDO = False

def key(r):
    return (r["method"], r["seed"], r["src"])

if REDO:
    for t in TASKS:
        bad = {key(r) for r in runs[t] if r["collapsed"]}
        if not bad:
            continue
        print(t, "redoing:", sorted(bad))
        for fname in ("runs.jsonl", "results.jsonl"):
            f = OUT[t] / fname
            keep = [l for l in f.read_text().splitlines() if l.strip() and key(json.loads(l)) not in bad]
            f.write_text("\n".join(keep) + "\n")
        METHODS, out = " ".join(sorted({m for m, _, _ in bad})), OUT[t]
        !python src/token_baseline.py --task {t} --out {out} --methods {METHODS} --lr_full 1e-5 --lr_lora 1.5e-4

## 10. Reports

In [ ]:
from IPython.display import Markdown, display
for t in TASKS:
    out = OUT[t]
    !python src/token_report.py --task {t} --dir {out} > /dev/null
    display(Markdown((OUT[t] / "summary.md").read_text()))

## 11. Download the results

Unzip `ner_results.zip` into `results/ner/` and `pos_results.zip` into `results/pos/` in the repo on your laptop.
The same files also stay on Google Drive in `typology-aware-peft/results/`.

In [ ]:
from google.colab import files
for t in TASKS:
    shutil.make_archive(f"/content/{t}_results", "zip", OUT[t])
    files.download(f"/content/{t}_results.zip")